In [8]:
%%capture
!pip install -U openai pydantic langchain langchain-community langchain-openai

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter Your OpenAI API Key:")

Enter Your OpenAI API Key:··········


# Prompt Pipelining Overview


- 🧱 **Modularity**: Mix and match prompt pieces like LEGO blocks.
- 👓 **Readability**: Break down complex prompts into easy bits.
- 🧠**Flexibility**: Craft prompts on-the-go with logic-based assembly.
- 🔄 **Efficiency**: Loop to append for scenarios like few-shot learning.
- 📝✨**Hybrid Construction**: Combine fixed text with variable-filled templates for structure and spontaneity.
- 💬 **Chat-Friendly**: Create conversational prompts by stacking messages.
- 🛠️ **Customizability**: Let users build prompts with their own components.

**String Prompt Pipelining:**

- 🔗 **Sequential Flow**: Link templates or strings in order, starting with a prompt.

Prompt pipelining turns the art of prompt crafting into a modular, efficient process, perfect for those looking to streamline and enhance their prompt design. 💡


In [3]:
from langchain.prompts import PromptTemplate

prompt = (
    PromptTemplate.from_template("I'm heading to {destination}. ")
    + "Recommend a great {activity} spot!")

prompt

PromptTemplate(input_variables=['activity', 'destination'], input_types={}, partial_variables={}, template="I'm heading to {destination}. Recommend a great {activity} spot!")

In [4]:
prompt = prompt + "\n\nAlso, any local delicacies I should try?"
prompt

PromptTemplate(input_variables=['activity', 'destination'], input_types={}, partial_variables={}, template="I'm heading to {destination}. Recommend a great {activity} spot!\n\nAlso, any local delicacies I should try?")

In [5]:
print(prompt.template)

I'm heading to {destination}. Recommend a great {activity} spot!

Also, any local delicacies I should try?


In [6]:
prompt.format(destination="Punjab", activity="dining")

"I'm heading to Punjab. Recommend a great dining spot!\n\nAlso, any local delicacies I should try?"

### The key differences between prompt pipelining and multi-input prompts are:

🚂 **Composition**: Pipelining links individual prompts into a cohesive journey.

🖼️ **Independence**: Each pipeline component is crafted separately before integration.

🚶‍♂️ **Sequence**: Pipelining lines up components, unlike multi-input prompts that handle inputs collectively.

📋 **Reuse**: Pipelining excels in reusing pieces; multi-input prompts manage multiple data points in one go.

📖 **Outcome**: Pipelining produces a single narrative; multi-input prompts generate a combined result.

🧱 **Construction**: Pipelining is about assembling prompts step by step, while multi-input prompts are about managing various inputs at once.

In short, pipelining is like creating a melody note by note, whereas multi-input prompts are like playing chords, hitting multiple notes simultaneously.


# Use in a chain

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [ ]:
llm = ChatOpenAI(model="gpt-3.5-turbo-1106", temperature=0.75)

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

for chunk in chain.stream({"destination":"Punjab", "activity":"dining"}):
  print(chunk, end="", flush=True)

One popular dining spot in Punjab is Kesar Da Dhaba in Amritsar, known for its authentic Punjabi cuisine and traditional ambience. 

As for local delicacies, you should definitely try the famous Amritsari Kulcha, Makki di Roti and Sarson da Saag, and the rich and creamy Butter Chicken. Don't forget to also try the sweet Lassi and the refreshing Nimbu Pani (lemon water) to wash it all down!

In [10]:
prompt = prompt + " How should I greet the locals in a jolly, informal manner?"


In [11]:
prompt.template

"I'm heading to {destination}. Recommend a great {activity} spot!\n\nAlso, any local delicacies I should try? How should I greet the locals in a jolly, informal manner?"

In [16]:
llm = ChatOpenAI(model="gpt-3.5-turbo-1106", temperature=0.75)
output_parser = StrOutputParser()
chain = prompt | llm | output_parser

for chunk in chain.stream({"destination":"Punjab", "activity":"dining"}):
  print(chunk, end="", flush=True)

One great dining spot in Punjab is Kesar Da Dhaba in Amritsar. They are known for their delicious vegetarian Punjabi dishes and traditional ambiance.

Some local delicacies you should try include butter chicken, tandoori chicken, chole bhature, sarson da saag and makki di roti, and lassi.

To greet the locals in a jolly, informal manner, you can say "Sat Sri Akal" which means "blessings be with you" or "Sasriyakal" which means "hello." It's always nice to greet them with a smile and a warm handshake.

# Example usecase

In [17]:
class TravelChatbot:
    def __init__(self, base_template):
        self.model = ChatOpenAI(model="gpt-3.5-turbo-1106", temperature=0.75)
        self.base_prompt = PromptTemplate.from_template(base_template)

    def append_to_prompt(self, additional_text):
        self.base_prompt += additional_text

    def run_chain(self, destination, activity):
        output_parser = StrOutputParser()
        chain = self.base_prompt | self.model | output_parser
        for chunk in chain.stream({"destination":destination, "activity":activity}):
          print(chunk, end="", flush=True)

# Usage
base_template = "I'm heading to {destination}. Recommend a great {activity} spot!"

chatbot = TravelChatbot(base_template)

# Basic prompt
chatbot.run_chain(destination="Punjab", activity="dining")

One popular dining spot in Punjab is Barbeque Nation. They are known for their delicious barbeque dishes and a wide variety of food options. Another great choice is Kesar Da Dhaba, a famous traditional Punjabi restaurant known for its authentic Punjabi cuisine and warm hospitality. Enjoy your dining experience in Punjab!

In [18]:
# Append more to the prompt and run again
chatbot.append_to_prompt("\n\nAlso, any local delicacies I should try?")

chatbot.run_chain(destination="Punjab", activity="dining")

One highly recommended dining spot in Punjab is Bharawan Da Dhaba in Amritsar. They are known for their traditional Punjabi cuisine and offer a wide range of delicious dishes.

Some local delicacies you should try in Punjab include:

1. Makki di Roti and Sarson da Saag - a traditional Punjabi dish made with corn flour bread and mustard greens.

2. Chole Bhature - a popular North Indian dish consisting of spicy chickpeas and fried bread.

3. Tandoori Chicken - marinated chicken cooked in a tandoor (clay oven) for a smoky, flavorful taste.

4. Lassi - a traditional Punjabi drink made with yogurt, sugar, and spices, often served in a tall glass and garnished with a dollop of cream.

Enjoy your trip to Punjab and the delicious local cuisine!

In [19]:
chatbot.append_to_prompt(" How should I greet the locals in a friendly, informal, jolly colloquial manner?")

chatbot.run_chain(destination="Punjab", activity="dining")

One great dining spot in Punjab is Barbeque Nation, known for its delicious grilled meats and wide variety of buffet options.

Some local delicacies to try in Punjab include sarson da saag (mustard greens curry), makki di roti (cornbread), and butter chicken. Don't forget to also try lassi, a traditional Punjabi yogurt-based drink.

To greet locals in a friendly, informal, jolly colloquial manner, you can use the Punjabi greeting "Sat Sri Akal" which means "God is the ultimate truth" and is commonly used to say hello. You can also say "Ki haal hai?" which means "How are you?" in Punjabi. This is sure to bring a smile to their faces!

# Chat Prompt Pipeline

🧩 **Composition**: Chat prompt pipelining turns reusable message blocks into a complete conversation flow.

🛠️ **Versatility**: Mix and match static messages with dynamic templates for a custom dialogue.

🔗 **End Result**: You get a ChatPromptTemplate that's ready for action, crafted from your message lineup.

🏗️ **Modularity**: Like using building blocks, this method lets you construct prompts piece by piece for maximum flexibility.

In essence, chat prompt pipelining is about assembling conversations from logical blocks, creating a user-friendly and adaptable ChatPromptTemplate.


In [20]:
from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Setting the scene with a Cockney-themed system message
prompt = SystemMessage(content="Welcome to the East End Cockney Chat! 🇬🇧")

# Constructing a chat flow with dry humour
new_prompt = (
    prompt
    + HumanMessage(content="Alright, guv'nor?")
    + AIMessage(content="Not too shabby. Did you hear about the London fog?")
    + "{input}"
)

# Formatting the chat with the user's response
new_prompt.format_messages(input="No, what about it?")

print(new_prompt)


input_variables=['input'] input_types={} partial_variables={} messages=[SystemMessage(content='Welcome to the East End Cockney Chat! 🇬🇧', additional_kwargs={}, response_metadata={}), HumanMessage(content="Alright, guv'nor?", additional_kwargs={}, response_metadata={}), AIMessage(content='Not too shabby. Did you hear about the London fog?', additional_kwargs={}, response_metadata={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]


In [21]:
new_prompt.messages

[SystemMessage(content='Welcome to the East End Cockney Chat! 🇬🇧', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="Alright, guv'nor?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Not too shabby. Did you hear about the London fog?', additional_kwargs={}, response_metadata={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]

In [22]:
model = ChatOpenAI(model="gpt-3.5-turbo-1106", temperature=0.75)

output_parser = StrOutputParser()

chain = new_prompt | model | output_parser

# Running the chatbot to get the punchline
for chunk in chain.stream({"input":"No, what about it?"}):
  print(chunk, end="", flush=True)

It's been so thick lately that I can't see me hand in front of me face!

# Prompt Composition


🧬 **Prompt Composition**: Reuse prompt segments with ease using the PipelinePrompt feature.

🏁 **1. Final Prompt**: The end product that you present to the model.

🔗 **2. Pipeline Prompts**: A sequence of named prompt templates that pass information forward, each influencing the next.

To summarize, PipelinePrompt allows for the efficient building of complex prompts by reusing and chaining together smaller, named components.


In [23]:
from langchain.prompts.pipeline import PipelinePromptTemplate
from langchain.prompts.prompt import PromptTemplate

In [24]:
full_template = """{introduction}

{example}

{start}"""

full_prompt = PromptTemplate.from_template(full_template)

In [25]:
full_prompt.input_variables

['example', 'introduction', 'start']

In [26]:
introduction_template = """You are impersonating {person}."""

introduction_prompt = PromptTemplate.from_template(introduction_template)

In [27]:
introduction_prompt.input_variables

['person']

In [28]:
example_template = """Here's an example of an interaction:

Q: {example_q}
A: {example_a}"""

example_prompt = PromptTemplate.from_template(example_template)

In [29]:
start_template = """Now, do this for real!

Q: {input}
A:"""

start_prompt = PromptTemplate.from_template(start_template)

In [30]:
full_prompt

PromptTemplate(input_variables=['example', 'introduction', 'start'], input_types={}, partial_variables={}, template='{introduction}\n\n{example}\n\n{start}')

In [31]:
input_prompts = [
    ("introduction", introduction_prompt),
    ("example", example_prompt),
    ("start", start_prompt)
]
pipeline_prompt = PipelinePromptTemplate(final_prompt=full_prompt, pipeline_prompts=input_prompts)

/tmp/ipython-input-31-2467874530.py:6: LangChainDeprecationWarning: This class is deprecated. Please see the docstring below or at the link for a replacement option: https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.pipeline.PipelinePromptTemplate.html
  pipeline_prompt = PipelinePromptTemplate(final_prompt=full_prompt, pipeline_prompts=input_prompts)


In [32]:
pipeline_prompt.input_variables

['input', 'person', 'example_a', 'example_q']

In [33]:
last_prompt = pipeline_prompt.format(
    person="Elon Musk",
    example_q="What's your favorite car?",
    example_a="Tesla",
    input="What's your favorite social media site?"
)

print(last_prompt)

You are impersonating Elon Musk.

Here's an example of an interaction:

Q: What's your favorite car?
A: Tesla

Now, do this for real!

Q: What's your favorite social media site?
A:


In [34]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-1106", temperature=0.75)

chain = pipeline_prompt | llm | StrOutputParser()

chain.invoke({
    "person":"Elon Musk","example_q":"What's your favorite car?",
    "example_a":"Tesla",
    "input":"What's your favorite social media site and why?"
    })

'Twitter. I find it to be a great platform for sharing updates on my projects, engaging with fans and followers, and even making important announcements. Plus, the character limit forces me to be concise and to the point.'

In [35]:
from langchain.prompts import PromptTemplate, PipelinePromptTemplate

class CookingShowChatbot:
    def __init__(self):
        # Base template for the cooking show scenario
        self.full_template = """
        {introduction}

        {example_dish}

        {present_dish}"""
        self.full_prompt = PromptTemplate.from_template(self.full_template)

        # Introduction where the user impersonates a famous chef
        self.introduction_template = """Welcome to the cooking show! Today, you're channeling the spirit of Chef {chef_name}."""
        self.introduction_prompt = PromptTemplate.from_template(self.introduction_template)

        # Example dish made by the famous chef
        self.example_dish_template = """Remember when Chef {chef_name} made that delicious {example_dish_name}? It was a hit!"""
        self.example_dish_prompt = PromptTemplate.from_template(self.example_dish_template)

        # User's turn to present their dish
        self.present_dish_template = """Now, it's your turn! Show us how you make your {user_dish_name}. Let's get cooking!"""
        self.present_dish_prompt = PromptTemplate.from_template(self.present_dish_template)

        # Combining the prompts into a pipeline
        self.input_prompts = [
            ("introduction", self.introduction_prompt),
            ("example_dish", self.example_dish_prompt),
            ("present_dish", self.present_dish_prompt)
        ]
        self.pipeline_prompt = PipelinePromptTemplate(final_prompt=self.full_prompt,
                                                      pipeline_prompts=self.input_prompts
                                                      )

    def run_scenario(self, chef_name, example_dish_name, user_dish_name):
        chain = self.pipeline_prompt | llm | StrOutputParser()

        response = chain.invoke({"chef_name":chef_name, "example_dish_name":example_dish_name, "user_dish_name":user_dish_name})

        return response




In [36]:
chatbot = CookingShowChatbot()
scenario = chatbot.run_scenario(chef_name="Gordon Ramsay", example_dish_name="Beef Wellington", user_dish_name="Vegan Lasagna")
print(scenario)

First, let's start by preheating the oven to 375°F (190°C).

Next, heat some olive oil in a large skillet over medium heat. Add diced onions, minced garlic, and chopped mushrooms. Sauté until the vegetables are softened.

Add in some diced tomatoes, tomato sauce, and tomato paste. Stir in some dried basil, oregano, salt, and pepper. Let the sauce simmer for about 10 minutes.

While the sauce is simmering, let's prepare our vegan "cheese" filling. In a bowl, mix together some tofu, nutritional yeast, garlic powder, and salt. Set this aside for now.

Now it's time to assemble the lasagna! Spread a thin layer of the tomato sauce on the bottom of a baking dish. Place lasagna noodles on top of the sauce, followed by a layer of the tofu "cheese" mixture and some fresh spinach.

Repeat the layers until you've used up all the ingredients, finishing with a layer of sauce on top.

Cover the baking dish with aluminum foil and bake in the preheated oven for 40 minutes. Then, remove the foil and ba